# 01 - Data cleaning & the SQL layer

**Goal:** turn the raw IBM Telco extract (7,043 customers, 21 columns) into a typed, validated
analytical table - and make the cleaning rules auditable by writing them in SQL.

Design choice: the raw CSV is loaded into SQLite *as text*. All typing and cleaning happens in
`sql/01_create_schema.sql`, and `sql/02_data_quality_checks.sql` gates the pipeline. The same
SQL later cleans any file uploaded to the dashboard, so training and scoring can never drift apart.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Image, display
from src import config
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

In [2]:
from src.data_preprocessing import load_raw_csv, build_database, run_quality_checks, run_named_query
raw = load_raw_csv()
print(raw.shape)
raw.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.7,151.65,Yes


## Finding 1 - `TotalCharges` has blanks, and they are not random

In [3]:
blank = raw["TotalCharges"].str.strip() == ""
print(f"blank TotalCharges: {blank.sum()}")
raw.loc[blank, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

blank TotalCharges: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20,,No
5218,2923-ARZLG,0,19.7,,No
6670,4075-WKNIU,0,73.35,,No


Every blank belongs to a customer with **tenure = 0**: they signed up but have not been billed yet.
The correct value is therefore **0.0**, not a median imputation (which would invent ~$1,400 of
history for brand-new customers). Pandas would have silently parsed these as NaN - reading the
file as text first is what made the pattern visible.

## Finding 2 - duplicates, categories and the 'No internet service' level

In [4]:
print("duplicate customer IDs:", raw["customerID"].duplicated().sum())
for col in ["MultipleLines", "OnlineSecurity", "StreamingTV", "Contract", "PaymentMethod"]:
    print(f"{col:<16}", raw[col].value_counts().to_dict())

duplicate customer IDs: 0
MultipleLines    {'No': 3390, 'Yes': 2971, 'No phone service': 682}
OnlineSecurity   {'No': 3498, 'Yes': 2019, 'No internet service': 1526}
StreamingTV      {'No': 2810, 'Yes': 2707, 'No internet service': 1526}
Contract         {'Month-to-month': 3875, 'Two year': 1695, 'One year': 1473}
PaymentMethod    {'Electronic check': 2365, 'Mailed check': 1612, 'Bank transfer (automatic)': 1544, 'Credit card (automatic)': 1522}


`No internet service` / `No phone service` are not a third state: the parent column
(`InternetService`, `PhoneService`) already carries that information. The SQL collapses them to
`No`, which removes 7 redundant one-hot columns without losing anything.

## The cleaning SQL

In [5]:
sql = (config.SQL_DIR / "01_create_schema.sql").read_text()
print(sql[sql.index("CREATE TABLE customers"):sql.index("CREATE INDEX")])

CREATE TABLE customers AS
SELECT
    customerID                                          AS customer_id,
    gender,
    CAST(SeniorCitizen AS INTEGER)                      AS senior_citizen,
    Partner                                             AS partner,
    Dependents                                          AS dependents,
    CAST(tenure AS INTEGER)                             AS tenure,
    PhoneService                                        AS phone_service,

    -- "No phone service" / "No internet service" are not a third state: they
    -- are a "No" that carries the parent subscription's information, which the
    -- parent column already holds. Collapsing them avoids inventing a category.
    CASE WHEN MultipleLines    = 'No phone service'    THEN 'No' ELSE MultipleLines    END AS multiple_lines,
    InternetService                                     AS internet_service,
    CASE WHEN OnlineSecurity   = 'No internet service' THEN 'No' ELSE OnlineSecurity   END AS online_

## Data-quality gate

In [6]:
build_database()           # raises DataQualityError if any check fails
import sqlite3
with sqlite3.connect(config.DB_PATH) as con:
    checks = run_quality_checks(con)
checks

,check_name,value,violations
0,row_count,7043,0
1,duplicate_customer_ids,0,0
2,null_monthly_charges,0,0
3,negative_or_zero_charges,0,0
4,tenure_out_of_range,0,0
5,churn_not_binary,0,0
6,zero_tenure_with_charges,0,0
7,total_charges_implausible,0,0
8,orphan_internet_addons,0,0


**Investigated, not suppressed.** The first version of `total_charges_implausible` (total must be
within 0.5-1.5x of tenure x monthly charges) flagged two customers. Both had tenure 2-3 months
with a ratio of 1.53-1.57: with so few bills, one plan change in month one explains it. They are
real accounts, so the check now allows one month of slack instead of the rows being dropped.

In [7]:
run_named_query("kpi_overview")  # a first look through the BI layer

,customers,churned,churn_rate_pct,monthly_revenue,monthly_revenue_lost,revenue_churn_pct,avg_tenure_months
0,7043,1869,26.54,456116.6,139130.85,30.5,32.4


## Outliers

In [8]:
feats = pd.read_csv(config.FEATURES_CSV)
rows = []
for col in ["tenure", "monthly_charges", "total_charges"]:
    q1, q3 = feats[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n = ((feats[col] < q1 - 1.5 * iqr) | (feats[col] > q3 + 1.5 * iqr)).sum()
    rows.append({"column": col, "min": feats[col].min(), "max": feats[col].max(), "iqr_outliers": n})
pd.DataFrame(rows)

,column,min,max,iqr_outliers
0,tenure,0.00,72.00,0
1,monthly_charges,18.25,118.75,0
2,total_charges,0.00,8684.80,0


No IQR outliers: tenure is capped at 72 months and charges are bounded by the price list.
`total_charges` is right-skewed (it is roughly tenure x price), which is why tree models and
standardised logistic regression handle it without a log transform.

## Class balance

In [9]:
feats["churn"].value_counts(normalize=True).rename("share")

churn
0    0.73463
1    0.26537
Name: share, dtype: float64

**26.5% churn.** Imbalanced but not extreme. A model that predicts "stays" for everyone would be
73.5% accurate, which is why this project judges models on ROC-AUC, PR-AUC and money instead.